![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 16 -- Lab 2: Homework ECG Beats with a Dataset Class

Homework. Day 10's heartbeat task again: 150 numbers per beat, five classes, 70% of them normal, scored by balanced accuracy. On Day 10 the rules allowed four hand-made numbers and a ten-tree forest (0.46). Now you feed the raw 150 numbers to a network through a Dataset class of your own, train it with the loop of today, keep the best epoch by balanced accuracy (not by accuracy and not by the loss), estimate the score with stratified 5-fold cross-validation, and look at the gap between the training and validation curves. Day 17's Lab 1 starts from this notebook and closes that gap.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The Polish AI Olympiad 2025, stage 1, task 3 (ECG): `train_validation_sets.npz` read from the OlimpiadaAI GitHub repository in the setup cell: 2,000 training beats and 1,500 validation beats, 150 numbers each, labels 0 (normal) to 4.

---
# Setup

In [ ]:
# numpy, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy matplotlib scikit-learn torch

In [ ]:
import io, copy, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
torch.manual_seed(0)

In [ ]:
URL = "https://raw.githubusercontent.com/OlimpiadaAI/II-OlimpiadaAI/main/1_etap/3_wykrywanie_zaburzen_sygnalu_ekg/train_validation_sets.npz"
d = np.load(io.BytesIO(urllib.request.urlopen(URL).read()))
X_train, y_train = d["X_train"].astype(np.float32), d["y_train"].astype(int)
X_val, y_val = d["X_validation"].astype(np.float32), d["y_validation"].astype(int)
CLASSES = ["normal", "afib", "pac", "pvc", "st_elev"]
print(X_train.shape, X_val.shape, np.bincount(y_train))

---
# Part 1 -- The beats and the baseline

## Task 1: Look at the beats

Plot one beat of each class (the first row with that label) as a line, 5 subplots side by side, the class name as the title. Then print the share of class 0 in `y_train` (expected 0.70).

In [ ]:
# Your code here


## Task 2: The baseline everyone must beat

Compute the balanced accuracy of the model that calls every beat normal: `balanced_accuracy_score(y_val, np.zeros_like(y_val))`. Expected 0.2. Day 10's forest on four hand-made numbers reached 0.46.

In [ ]:
# Your code here


---
# Part 2 -- A Dataset class

The *Definition: Dataset and DataLoader* and *Your Own Dataset Class* slides. The standardisation statistics come from the training beats only (Day 09's leakage rule).

## Task 3: The Dataset class

Write `class BeatDataset(Dataset)` with `__init__(self, X, y, mean, std)` that stores `self.X = torch.tensor((X - mean) / std, dtype=torch.float32)` and `self.y = torch.tensor(y)`; `__len__` returning the number of beats; `__getitem__(self, i)` returning `(self.X[i], self.y[i])`. Compute `mean = X_train.mean(axis=0)` and `std = X_train.std(axis=0)`, set `std[std < 1e-6] = 1.0` (a constant column must not be divided by 0), build `train_ds = BeatDataset(X_train, y_train, mean, std)` and `val_ds = BeatDataset(X_val, y_val, mean, std)`, and print `len(train_ds)` and the shape and dtypes of `train_ds[0]`.

Syntax hint (the shape of the call, not the answer):

```python
class MyData(Dataset):
    def __init__(self, ...):   # store what one sample needs
    def __len__(self):         # how many samples
    def __getitem__(self, i):  # return (x_i, y_i) as tensors
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
x0, y0 = train_ds[0]
assert len(train_ds) == 2000 and x0.shape == (150,) and x0.dtype == torch.float32 and y0.dtype == torch.int64, 'BeatDataset: standardised float32 features, int64 labels, one pair per index'
print('Task 3 passed')

## Task 4: The loaders

Build `train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)` and `val_loader = DataLoader(val_ds, batch_size=500)`. Print `len(train_loader)` (32) and the shapes of the first training batch.

In [ ]:
# Your code here


---
# Part 3 -- The network and the loop

The *Loop in Full* and *Validation Inside the Loop* slides. The checkpoint follows the task's metric: balanced accuracy.

## Task 5: The network

Write `make_model()` returning `nn.Sequential(nn.Linear(150, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 5))`. Create `model = make_model()`, print the parameter count (expected 27,909), and check that `model(xb).shape` is `(64, 5)`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert sum(p.numel() for p in model.parameters()) == 27909 and model(xb).shape == (64, 5), 'make_model: 150-128-64-5 with ReLU between the layers'
print('Task 5 passed')

## Task 6: Evaluate by the task's metric

Write `evaluate(model, loader)` returning `(mean_loss, balanced_acc)`: eval mode, `no_grad`, loop over the loader, sum the losses weighted by batch size, collect all predictions (`argmax`) and labels, then `balanced_accuracy_score(labels, preds)`. Use `loss_fn = nn.CrossEntropyLoss()`. Print the untrained model's numbers on `val_loader`: loss about 1.6 (ln 5), balanced accuracy about 0.2.

In [ ]:
# Your code here


## Task 7: The loop with a checkpoint on balanced accuracy

Write `fit(model, train_loader, val_loader, epochs, lr)`: SGD with `lr`; per epoch the four moves over the training batches (mean training loss), then `evaluate` on validation; keep `best_state = copy.deepcopy(model.state_dict())` whenever the **balanced accuracy** improves (and remember `best_score, best_epoch`); return `history` (list of `(train_loss, val_loss, val_bal)`) and `(best_score, best_epoch, best_state)`. Run it with `model = make_model()`, 60 epochs, lr 0.05 (about 20 seconds). Expected best balanced accuracy about 0.94, reached late in the run.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_score > 0.85 and len(history) == 60, 'fit: 60 epochs of SGD lr 0.05, best state kept by validation balanced accuracy'
print('Task 7 passed')

## Task 8: The gap

Plot the training loss and the validation loss per epoch on the left, and the validation balanced accuracy on the right, with a vertical line at `best_epoch`. Where does the training loss end, and what does the validation loss do after epoch 20? This gap is what Day 17 closes.

In [ ]:
# Your code here


---
# Part 4 -- Stratified 5-fold cross-validation

The folds of Day 13: stratified, so every fold keeps the 70 / 7.5 / 7.5 / 7.5 / 7.5 class shares. The standardisation is recomputed inside each fold from its training part.

## Task 9: Five folds

Loop over `StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X_train, y_train)`. For each fold: compute `mean, std` from `X_train[tr]` (constant columns set to 1), build `BeatDataset`s and loaders for `tr` and `va`, create `make_model()` after `torch.manual_seed(0)`, run `fit` for 40 epochs with lr 0.05, and store the fold's best balanced accuracy and best epoch in two lists `fold_scores, fold_epochs`. Print the five scores, their mean and standard deviation. Expected mean about 0.87 with a spread of about 0.02.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(fold_scores) == 5 and 0.7 < np.mean(fold_scores) < 0.97, 'folds: StratifiedKFold(5, shuffle=True, random_state=0), fit 40 epochs per fold'
print('Task 9 passed')

## Task 10: The official validation set, once

Load `best_state` from Task 7 into a fresh `make_model()` (`load_state_dict`), evaluate it on `val_loader`, and print the balanced accuracy and `confusion_matrix(y_val, preds)` (collect the predictions as in `evaluate`). Which class is missed most, and how does the official score compare with the fold mean?

In [ ]:
# Your code here


---
## Task 11: Reflect

The checkpoint followed balanced accuracy. What would have gone wrong with plain accuracy on this table, and with the validation loss? Why must the standardisation be recomputed inside every fold rather than once on all 2,000 beats? Describe the gap you plotted in Task 8 in one sentence, and name three dials from Day 17's topics that you expect to close it.

*Your answers here*

---
## Conclusion

- **A Dataset class**: three methods turned 2,000 beats into standardised `(x, y)` pairs that a DataLoader could batch and shuffle.
- **The loop with the right checkpoint**: the best epoch was chosen by the task's metric, balanced accuracy, and its dials kept with `state_dict()`.
- **Stratified 5-fold**: five scores with a mean and a spread, every fold keeping the class shares, every fold standardised on its own training part.
- **The gap**: the training loss went towards zero while the validation loss turned; the raw numbers beat Day 10's forest by far, and Day 17 shows how to close the gap.

---

Made By **Sattam Altwaim**